# Laboratorio 4 — Ejercicio 1

## Problema de Programación Lineal resuelto por Simplex y por Pulp

Considere el siguiente problema de programación lineal:

$$\text{Maximizar } z = 120x_1 + 80x_2 + 100x_3 + 150x_4,$$

sujeto a

$$
\begin{aligned}
4x_1 + 1x_2 + 2x_3 + 3x_4 &\leq 60 \\
2x_1 + 2x_2 + x_3 + 3x_4 &\leq 48 \\
x_1 + x_2 + 3x_3 + 2x_4 &\leq 36 \\
3x_1 + x_2 + 2x_3 + 4x_4 &\leq 30 \\
x_1, x_2, x_3, x_4 &\geq 0.
\end{aligned}
$$

**a)** Resolver el problema implementando el algoritmo simplex en Excel o realizado a mano. Dejar constancia del procedimiento completo.

**b)** Comparar con la solución obtenida en JuMP o Pulp.

El inciso (a) se desarrolló en el archivo **`Ejercicio_1_Simplex.xlsx`** de esta misma carpeta, donde se documenta el procedimiento completo del método simplex: la forma estándar, cada uno de los tableaux con su columna pivote, su prueba de la razón mínima y su elemento pivote resaltados, y el tableau final con la lectura de la solución óptima. Este notebook desarrolla el inciso (b).

## Resumen del ejercicio

El **método simplex** resuelve un programa lineal recorriendo los vértices de la región factible. Se parte de una solución básica factible —en un problema de maximización con todas las restricciones de tipo $\le$ y lado derecho no negativo, el origen, con las variables de holgura como base— y en cada iteración se intercambia una variable de la base por otra, moviéndose a un vértice adyacente que no empeora el valor objetivo. La variable que **entra** se escoge por el criterio de Dantzig (el coeficiente más negativo de la fila $z$, es decir, la que más rápido incrementa $z$ por unidad) y la que **sale** por la prueba de la razón mínima, que identifica la restricción que primero se satura y así preserva la factibilidad. El algoritmo termina cuando ningún coeficiente de la fila $z$ es negativo: en ese punto no existe variable no básica que al entrar mejore el objetivo, y el vértice alcanzado es óptimo.

En este notebook el mismo problema se resuelve con **Pulp**, que construye el modelo de forma declarativa y lo entrega al solver CBC. El propósito no es solo obtener la respuesta, sino **contrastarla contra el simplex hecho a mano en Excel**: si ambos procedimientos —uno ejecutado paso a paso sobre los tableaux y otro por un solver profesional— llegan al mismo vértice y al mismo valor de $z$, queda validado el desarrollo del inciso (a).

## Objetivo y referencias

Se resolverá el programa lineal con la librería **Pulp**, que utiliza como motor de optimización el solver **CBC** (*COIN-OR Branch and Cut*), y se compararán sus resultados con los del simplex implementado en Excel: la solución óptima, el valor de la función objetivo, las holguras de cada restricción y los precios sombra.

El desarrollo se apoya en `pulp` para el modelado y la solución, `numpy` para el manejo matricial de los coeficientes y la verificación de las restricciones, y `pandas` para presentar las tablas comparativas.

## 1. Datos del problema

Se cargan los coeficientes de la función objetivo, la matriz de restricciones y el vector de lados derechos tal como aparecen en el enunciado.

In [1]:
import numpy as np
import pandas as pd
import pulp

# Coeficientes de la función objetivo: z = 120x1 + 80x2 + 100x3 + 150x4
c = np.array([120, 80, 100, 150])

# Matriz de coeficientes de las restricciones (una fila por restricción).
A = np.array([
    [4, 1, 2, 3],
    [2, 2, 1, 3],
    [1, 1, 3, 2],
    [3, 1, 2, 4],
])

# Lados derechos (recursos disponibles).
b = np.array([60, 48, 36, 30])

n_var = len(c)
n_res = len(b)

modelo_tabla = pd.DataFrame(A, columns=[f"x{j+1}" for j in range(n_var)],
                            index=[f"Restricción {i+1}" for i in range(n_res)])
modelo_tabla["Signo"] = "<="
modelo_tabla["RHS"] = b

print(f"Variables de decisión: {n_var}")
print(f"Restricciones: {n_res}")
print(f"Coeficientes de la función objetivo: {c}")
modelo_tabla

Variables de decisión: 4
Restricciones: 4
Coeficientes de la función objetivo: [120  80 100 150]


,x1,x2,x3,x4,Signo,RHS
Restricción 1,4,1,2,3,<=,60
Restricción 2,2,2,1,3,<=,48
Restricción 3,1,1,3,2,<=,36
Restricción 4,3,1,2,4,<=,30


### Explicación de esta parte:

`c` contiene los coeficientes de la función objetivo y `A` la matriz de coeficientes tecnológicos, con una fila por restricción y una columna por variable de decisión; `b` recoge los recursos disponibles. Guardar el modelo en forma matricial permite escribir la formulación en Pulp con un solo ciclo y, más adelante, verificar el cumplimiento de las restricciones evaluando directamente el producto $Ax$ sin volver a teclear los coeficientes, lo que evita errores de transcripción.

## 2. Construcción y solución del modelo en Pulp (inciso b)

Se declara un problema de maximización, se agregan las cuatro variables de decisión con cota inferior cero —que es como Pulp expresa la restricción de no negatividad $x_j \ge 0$— y se incorporan las cuatro restricciones del enunciado.

In [2]:
modelo = pulp.LpProblem("Ejercicio_1_maximizacion", pulp.LpMaximize)

# Variables de decisión no negativas y continuas.
x = [pulp.LpVariable(f"x{j+1}", lowBound=0, cat="Continuous") for j in range(n_var)]

# Función objetivo: z = 120x1 + 80x2 + 100x3 + 150x4
modelo += pulp.lpDot(c, x), "z"

# Restricciones del enunciado.
for i in range(n_res):
    modelo += pulp.lpDot(A[i], x) <= b[i], f"Restriccion_{i+1}"

modelo

Ejercicio_1_maximizacion:
MAXIMIZE
120*x1 + 80*x2 + 100*x3 + 150*x4 + 0.0
SUBJECT TO
Restriccion_1: 4 x1 + x2 + 2 x3 + 3 x4 <= 60

Restriccion_2: 2 x1 + 2 x2 + x3 + 3 x4 <= 48

Restriccion_3: x1 + x2 + 3 x3 + 2 x4 <= 36

Restriccion_4: 3 x1 + x2 + 2 x3 + 4 x4 <= 30

VARIABLES
x1 Continuous
x2 Continuous
x3 Continuous
x4 Continuous

### Explicación de esta parte:

`pulp.LpProblem` con el sentido `LpMaximize` declara el problema; cada `LpVariable` se crea con `lowBound=0`, que es exactamente la condición de no negatividad del enunciado, y como `Continuous`, tal como pide el inciso. La expresión `pulp.lpDot(c, x)` construye el producto punto $c^\top x$ de forma compacta, y el ciclo agrega una a una las restricciones $A_i x \le b_i$ con un nombre identificable. Al imprimir el objeto `modelo` se obtiene la formulación completa en texto, lo que permite verificar que lo que se le entrega al solver coincide exactamente con el enunciado.

In [3]:
modelo.solve(pulp.PULP_CBC_CMD(msg=0))

x_pulp = np.array([v.value() for v in x])
z_pulp = pulp.value(modelo.objective)

print(f"Estado del solver: {pulp.LpStatus[modelo.status]}")
print(f"\nSolución óptima:")
for j in range(n_var):
    print(f"  x{j+1} = {x_pulp[j]:.4f}")
print(f"\nValor óptimo de la función objetivo: z = {z_pulp:,.4f}")

Estado del solver: Optimal

Solución óptima:
  x1 = 0.0000
  x2 = 22.0000
  x3 = 4.0000
  x4 = 0.0000

Valor óptimo de la función objetivo: z = 2,160.0000


### Explicación de esta parte:

El solver CBC devuelve el estado `Optimal`, lo que confirma que el problema es factible, acotado, y que la solución hallada es óptima global —en un programa lineal la región factible es convexa, de modo que todo óptimo local también es global. La solución es $x = (0,\ 22,\ 4,\ 0)$ con $z = 2160$: conviene concentrar la producción en las variables $x_2$ y $x_3$ y dejar $x_1$ y $x_4$ en cero, a pesar de que $x_4$ es la de mayor coeficiente en la función objetivo (150). La razón es que $x_4$ consume demasiado de los recursos escasos —requiere 4 unidades de la cuarta restricción, cuyo total es apenas 30— de modo que su aporte por unidad de recurso consumido resulta menor que el de $x_2$ y $x_3$.

## 3. Verificación de las restricciones y holguras

Se comprueba que la solución entregada por Pulp satisface efectivamente las cuatro restricciones, y se calcula la holgura de cada una para identificar qué recursos quedan saturados en el óptimo.

In [4]:
lhs = A @ x_pulp          # Consumo de cada recurso en el óptimo
holguras = b - lhs        # Recurso disponible que queda sin usar

verificacion = pd.DataFrame({
    "Restricción": [f"Restricción {i+1}" for i in range(n_res)],
    "LHS (consumo)": lhs,
    "RHS (disponible)": b,
    "Holgura": holguras,
    "¿Se cumple?": ["Sí" if lhs[i] <= b[i] + 1e-9 else "No" for i in range(n_res)],
    "Estado": ["activa (saturada)" if abs(holguras[i]) < 1e-9 else "inactiva (sobra recurso)"
               for i in range(n_res)],
})

print(f"¿Todas las restricciones se cumplen? {bool((lhs <= b + 1e-9).all())}")
print(f"¿Todas las variables son no negativas? {bool((x_pulp >= -1e-9).all())}")
verificacion

¿Todas las restricciones se cumplen? True
¿Todas las variables son no negativas? True


,Restricción,LHS (consumo),RHS (disponible),Holgura,¿Se cumple?,Estado
0,Restricción 1,30.0,60,30.0,Sí,inactiva (sobra recurso)
1,Restricción 2,48.0,48,0.0,Sí,activa (saturada)
2,Restricción 3,34.0,36,2.0,Sí,inactiva (sobra recurso)
3,Restricción 4,30.0,30,0.0,Sí,activa (saturada)


### Explicación de esta parte:

El producto $Ax$ da el consumo de cada recurso en el óptimo y su diferencia con $b$ es la holgura, que corresponde exactamente al valor de las variables $s_i$ del tableau final del Excel. Las restricciones 2 y 4 quedan **activas**: se consumen íntegramente sus 48 y 30 unidades, de modo que son ellas las que limitan el valor de $z$. Las restricciones 1 y 3 quedan **inactivas**, con 30 y 2 unidades sobrantes respectivamente; si se relajaran no se ganaría nada, porque no son las que están frenando la solución. Esta lectura es coherente con el hecho de que el simplex terminó con $s_1 = 30$ y $s_3 = 2$ en la base y con $s_2 = s_4 = 0$ fuera de ella.

## 4. Comparación con el simplex implementado en Excel (inciso b)

El archivo `Ejercicio_1_Simplex.xlsx` documenta el desarrollo completo del método simplex sobre este mismo problema. Partiendo de la base inicial $\{s_1, s_2, s_3, s_4\}$, el algoritmo realizó tres iteraciones:

| Iteración | Entra | Sale | Elemento pivote | Razón mínima | $z$ resultante |
|:---------:|:-----:|:----:|:---------------:|:------------:|:--------------:|
| 1 | $x_4$ | $s_4$ | 4    | $30/4 = 7.5$    | 1125 |
| 2 | $x_2$ | $s_2$ | 5/4  | $25.5/1.25 = 20.4$ | 1992 |
| 3 | $x_3$ | $x_4$ | 3/5  | $2.4/0.6 = 4$   | 2160 |

y terminó con la base $\{s_1, x_2, s_3, x_3\}$ y todos los coeficientes de la fila $z$ no negativos, es decir, en el óptimo. A continuación se contrastan numéricamente ambos resultados.

In [5]:
# Solución leída del tableau final del archivo Ejercicio_1_Simplex.xlsx
x_excel = np.array([0.0, 22.0, 4.0, 0.0])
holguras_excel = np.array([30.0, 0.0, 2.0, 0.0])
z_excel = 2160.0

comparacion = pd.DataFrame({
    "Variable": [f"x{j+1}" for j in range(n_var)] + [f"s{i+1}" for i in range(n_res)] + ["z"],
    "Simplex (Excel)": list(x_excel) + list(holguras_excel) + [z_excel],
    "Pulp (CBC)": list(x_pulp) + list(holguras) + [z_pulp],
})
comparacion["Diferencia"] = comparacion["Simplex (Excel)"] - comparacion["Pulp (CBC)"]

coinciden = (np.allclose(x_excel, x_pulp)
             and np.allclose(holguras_excel, holguras)
             and np.isclose(z_excel, z_pulp))
print(f"¿Coinciden ambas soluciones? {coinciden}")
print(f"Diferencia máxima en las variables: {np.abs(x_excel - x_pulp).max():.2e}")
print(f"Diferencia en la función objetivo: {abs(z_excel - z_pulp):.2e}")
comparacion

¿Coinciden ambas soluciones? True
Diferencia máxima en las variables: 0.00e+00
Diferencia en la función objetivo: 0.00e+00


,Variable,Simplex (Excel),Pulp (CBC),Diferencia
0,x1,0.0,0.0,0.0
1,x2,22.0,22.0,0.0
2,x3,4.0,4.0,0.0
3,x4,0.0,0.0,0.0
4,s1,30.0,30.0,0.0
5,s2,0.0,0.0,0.0
6,s3,2.0,2.0,0.0
7,s4,0.0,0.0,0.0
8,z,2160.0,2160.0,0.0


### Explicación de esta parte:

La comparación es exacta: ambos métodos entregan $x = (0,\ 22,\ 4,\ 0)$, las mismas holguras $s = (30,\ 0,\ 2,\ 0)$ y el mismo valor objetivo $z = 2160$, con diferencias del orden de $10^{-15}$ o nulas, atribuibles únicamente a la representación en punto flotante. Esto era esperable: el simplex desarrollado a mano en el Excel y el solver CBC recorren la región factible por caminos que pueden diferir, pero ambos convergen al mismo vértice óptimo, que en este problema es único. La coincidencia valida el procedimiento del inciso (a), incluyendo la elección de las variables de entrada y salida en cada iteración y las operaciones de pivoteo.

## 5. Precios sombra

En el tableau final del simplex, los coeficientes de la fila $z$ ubicados en las columnas de las variables de holgura son los **precios sombra** de las restricciones: indican en cuánto aumentaría $z$ si el lado derecho de esa restricción creciera en una unidad. Se comparan con los valores duales que reporta Pulp.

In [6]:
# Precios sombra leídos en la fila z del tableau final (columnas s1..s4) del Excel
precios_sombra_excel = np.array([0.0, 20.0, 0.0, 40.0])

# Valores duales que reporta Pulp para cada restricción
precios_sombra_pulp = np.array([modelo.constraints[f"Restriccion_{i+1}"].pi
                                for i in range(n_res)])

duales = pd.DataFrame({
    "Restricción": [f"Restricción {i+1}" for i in range(n_res)],
    "Precio sombra (Excel)": precios_sombra_excel,
    "Precio sombra (Pulp)": precios_sombra_pulp,
    "Holgura": holguras,
    "Interpretación": [
        (f"Una unidad adicional de recurso incrementa z en {p:.2f}." if p > 0
         else "Restricción no saturada: una unidad adicional no cambia z.")
        for p in precios_sombra_excel
    ],
})

print(f"¿Coinciden los precios sombra? "
      f"{np.allclose(precios_sombra_excel, precios_sombra_pulp)}")
duales

¿Coinciden los precios sombra? True


,Restricción,Precio sombra (Excel),Precio sombra (Pulp),Holgura,Interpretación
0,Restricción 1,0.0,-0.0,30.0,Restricción no saturada: una unidad adicional ...
1,Restricción 2,20.0,20.0,0.0,Una unidad adicional de recurso incrementa z e...
2,Restricción 3,0.0,-0.0,2.0,Restricción no saturada: una unidad adicional ...
3,Restricción 4,40.0,40.0,0.0,Una unidad adicional de recurso incrementa z e...


### Explicación de esta parte:

Los precios sombra obtenidos del tableau final del Excel coinciden con los valores duales que reporta Pulp: $(0,\ 20,\ 0,\ 40)$. Su lectura es directa y consistente con el análisis de holguras de la sección 3: las restricciones 1 y 3, que tienen holgura positiva, valen cero —sobra recurso, así que tener más no ayuda—, mientras que las restricciones 2 y 4, que están saturadas, tienen precio sombra positivo. La restricción 4 es la más valiosa: cada unidad adicional de ese recurso incrementaría la utilidad en \$40, contra \$20 de la restricción 2, de modo que es ahí donde conviene invertir si se quisiera ampliar la capacidad.

## 6. Conclusión

El problema fue resuelto por dos vías independientes y ambas llegaron al mismo resultado. El **simplex implementado en Excel** (inciso a, documentado en `Ejercicio_1_Simplex.xlsx`) alcanzó el óptimo en tres iteraciones, partiendo de la base de holguras $\{s_1, s_2, s_3, s_4\}$ e intercambiando sucesivamente $s_4$ por $x_4$, $s_2$ por $x_2$ y finalmente $x_4$ por $x_3$, hasta llegar a la base $\{s_1, x_2, s_3, x_3\}$ con todos los coeficientes de la fila $z$ no negativos. La **solución con Pulp** (inciso b) reproduce exactamente ese resultado:

$$x_1 = 0, \qquad x_2 = 22, \qquad x_3 = 4, \qquad x_4 = 0, \qquad z = 2160.$$

El acuerdo entre ambos métodos es exacto, tanto en los valores de las variables de decisión como en las holguras $(30, 0, 2, 0)$ y en los precios sombra $(0, 20, 0, 40)$, lo que valida el procedimiento manual del inciso (a). Es interesante notar que la solución óptima deja en cero a $x_4$, la variable con el mayor coeficiente en la función objetivo: lo que determina la solución no es el aporte unitario aislado de cada variable, sino su aporte relativo al consumo que hace de los recursos escasos, y $x_4$ resulta demasiado costosa en términos de la cuarta restricción, que es la más limitante del problema.